# Example - Identification of Model Parameters from Data

In [ ]:
import os

import casadi as cas
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from cas_models.discrete_time.estimate import estimate_params
from cas_models.discrete_time.models import StateSpaceModelDTARXSISO
from cas_models.discrete_time.simulate import (
    make_n_step_simulation_function_from_model,
)
from cas_models.transformations import hstack_systems
from plot_utils import make_io_plot

## Load Input-Output Data from Step Response Experiment

In [ ]:
data_dir = "data"
os.listdir(data_dir)

In [ ]:
filename = "data_center_sim_results.csv"
id_data = pd.read_csv(os.path.join(data_dir, filename))
id_data.info()

In [ ]:
id_data.head()

In [ ]:
input_names = ["CRAC1_flow", "CRAC2_flow"]
output_names = ["R11_intake"]

fig, axes = make_io_plot(id_data, input_names, output_names)
plt.tight_layout()
plt.show()

## Define the system model structure

In [ ]:
# Construct a 2-input, 1-output system model with symbolic parameters
sys1 = StateSpaceModelDTARXSISO(
    na=2, nb=2, nk=1, input_name="CRAC1_flow", output_name="R11_intake"
)
sys2 = StateSpaceModelDTARXSISO(
    na=2, nb=1, nk=1, input_name="CRAC2_flow", output_name="R11_intake"
)

# Sum the outputs of both systems into a combined model with one output
sys_model = hstack_systems([sys1, sys2], name="R11_intake", verbose_names=True)

sys_model.describe()

In [ ]:
u_nop = id_data[input_names].iloc[0].to_numpy()
y_nop = id_data[output_names].iloc[0].to_numpy()
u_nop, y_nop

In [ ]:
# Prepare input/output data
t = id_data.index.to_numpy()
nT = t.shape[0] - 1
simulate = make_n_step_simulation_function_from_model(sys_model, nT)
simulate

In [ ]:
# Initial guess of parameters
params_init = {
    "sys1_a_0": -1.6,
    "sys2_a_0": -1.8,
    "sys1_a_1": 0.61,
    "sys2_a_1": 0.81,
    "sys1_b_0": -0.95,
    "sys1_b_1": 0.87,
    "sys2_b": -0.027,
}

U = id_data[input_names].to_numpy() - u_nop
Y_m = id_data[output_names].to_numpy() - y_nop

x0 = cas.DM.zeros(sys_model.n, 1)
X, Y = simulate(
    cas.DM(t), U[:nT], x0, *[params_init[name] for name in sys_model.params]
)
X = np.asarray(X)
Y = np.asarray(Y)
assert X.shape == (nT + 1, sys_model.n)
assert Y.shape == (nT + 1, sys_model.ny)

y_pred = pd.DataFrame(
    Y + y_nop, index=id_data.index, columns=sys_model.output_names
)

fig, axes = make_io_plot(id_data, input_names, output_names, y_pred=y_pred)
plt.tight_layout()
plt.show()

y_pred_err = Y - Y_m
rmse = float(np.sqrt(np.mean(y_pred_err**2)))
print(f"{rmse = :.3f}")

In [ ]:
# Estimate the parameters by minimizing the output errors (multiple shooting)
params_est, x0_est, sol = estimate_params(
    sys_model,
    t,
    U[:-1, :],
    Y_m,
    n_steps=20,
    params_init=params_init,
    solver_options={"print_level": 0},
)
print(f"{sol.stats()["return_status"]} after {sol.stats()["iter_count"]} iterations")

In [ ]:
pd.concat(
    [
        pd.Series(params_init, name="params_init"),
        pd.Series(params_est, name="params_est"),
    ],
    axis=1,
)

In [ ]:
x0 = cas.DM.zeros(sys_model.n, 1)
X, Y = simulate(
    cas.DM(t), U[:nT], x0, *[params_est[name] for name in sys_model.params]
)
X = np.asarray(X)
Y = np.asarray(Y)
assert X.shape == (nT + 1, sys_model.n)
assert Y.shape == (nT + 1, sys_model.ny)

y_pred = pd.DataFrame(
    Y + y_nop, index=id_data.index, columns=sys_model.output_names
)

fig, axes = make_io_plot(id_data, input_names, output_names, y_pred=y_pred)
plt.tight_layout()
plt.show()

y_pred_err = Y - Y_m
rmse = float(np.sqrt(np.mean(y_pred_err**2)))

print(f"{rmse = :.3f}")